# Week 14, Day 1: Understanding the problem

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w014d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


> **Key idea.** Before you write code, you should be able to say, in your own words: what goes in, what comes out, and the rule that connects them. And you should have worked at least two examples by hand.

In [ ]:
def luhn_ok(number):
    digits = number.replace(" ", "")
    if len(digits) < 2 or not digits.isdigit():
        return False
    total = 0
    for i, ch in enumerate(reversed(digits)):
        d = int(ch)
        if i % 2 == 1:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0

print(luhn_ok("79927398713"))
print(luhn_ok("79927398710"))
print(luhn_ok("4539 1488 0343 6467"))

> **Tip.** When you find a gap, write your decision down as an example: `diff("23:00", "01:00") == 120`. Now the decision is visible, and it becomes a check.

In [ ]:
def clock_diff(start, end):
    """Minutes from start to end, going forward, so across midnight if needed."""
    def minutes(t):
        h, m = t.split(":")
        return int(h) * 60 + int(m)
    return (minutes(end) - minutes(start)) % (24 * 60)

assert clock_diff("09:00", "17:30") == 510
assert clock_diff("23:00", "01:00") == 120
assert clock_diff("17:30", "09:00") == 930
assert clock_diff("12:00", "12:00") == 0
print("all examples pass")

> **Key idea.** Understanding a problem means: inputs, outputs and rules written down; at least two examples worked by hand, including an awkward one; every gap you found decided and written as an example; and those examples turned into checks.

### Exercise: The check digit

The last digit of a card number is a **check digit**, chosen so the whole number passes the Luhn check. Write `check_digit(partial)` that takes the digits without the check digit (a string, no spaces) and returns the digit (as an int, 0 to 9) that makes it valid. For `"7992739871"` it's `3`. Work out why by hand first: when the check digit is added on the right, which digits get doubled?

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def luhn(n):\n    t = 0\n    for i, ch in enumerate(reversed(n)):\n        d = int(ch) * (2 if i % 2 else 1)\n        t += d - 9 if d > 9 else d\n    return t % 10 == 0\ndef test_values():\n    \"several\"\n    assert check_digit(\"7992739871\") == 3\n    for p in [\"453914880343646\", \"0\", \"12345\", \"999999999\"]:\n        d = check_digit(p)\n        assert isinstance(d, int) and 0 <= d <= 9 and luhn(p + str(d)), f\"check_digit({p!r}) gave {d}\"")

### Exercise: Look and say

In the **look-and-say** sequence, each term describes the previous one aloud: `1` is "one 1", so the next term is `11`; that's "two 1s", so `21`; then "one 2, one 1", so `1211`; then `111221`. Write `next_term(term)` (strings in, string out), and `look_say(n)` that returns the first `n` terms as a list, starting from `"1"`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert next_term(\"1\") == \"11\" and next_term(\"1211\") == \"111221\" and next_term(\"3\") == \"13\"\n    assert look_say(6) == [\"1\", \"11\", \"21\", \"1211\", \"111221\", \"312211\"]\n    assert look_say(0) == []")

### Exercise: Kaprekar's routine

Take a four-digit number whose digits aren't all the same (leading zeros allowed, so 0021 is fine). Arrange its digits in descending order and in ascending order, and subtract the smaller from the larger. Repeat with the result, always treating it as four digits (so 999 is 0999). The Indian mathematician D. R. Kaprekar found that this always reaches **6174** within 7 steps, and stays there. Write `kaprekar_steps(n)` that returns how many subtractions it takes to reach 6174 (0 for 6174 itself). Raise a `ValueError` if `n` isn't from 0 to 9999 or all its four digits are the same.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert kaprekar_steps(6174) == 0\n    assert kaprekar_steps(3524) == 3\n    assert kaprekar_steps(2111) == 5\n    assert kaprekar_steps(21) == 3 and kaprekar_steps(14) == 7\n    assert max(kaprekar_steps(n) for n in range(10000) if len(set(f\"{n:04d}\")) > 1) == 7\n    for bad in (1111, 0, 10000, -5):\n        try:\n            kaprekar_steps(bad)\n        except ValueError:\n            continue\n        raise AssertionError(f\"kaprekar_steps({bad}) should raise a ValueError\")")

### Exercise: ISBN-10

An old ISBN-10 book number has 10 characters: nine digits and a final check character, which is a digit or `X` (meaning 10). It's valid if 10 × the first + 9 × the second + ... + 1 × the last is a multiple of 11. Hyphens are ignored. Write `isbn10_ok(text)`. `"0-306-40615-2"` is valid. Anything else, wrong length, letters other than a final `X`, is invalid (return `False`).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    for t in [\"0-306-40615-2\", \"0306406152\", \"0-8044-2957-X\", \"080442957X\"]:\n        assert isbn10_ok(t) is True, t\n    for t in [\"0-306-40615-3\", \"030640615\", \"X306406152\", \"0-306-4O615-2\", \"\", \"0-8044-2957-x5\"]:\n        assert isbn10_ok(t) is False, t")

### Exercise: Examples first

Here's a description with gaps: *"Write `shorten(text, n)` that shortens text to at most n characters, adding '...' if it was cut."* The examples below decide the gaps. Study them, write down the rules they imply, then write the function.

```
shorten("Hello world", 20)      ->  "Hello world"      (fits: unchanged)
shorten("Hello world", 8)       ->  "Hello..."         (the result, dots included, is at most 8)
shorten("Hello world", 11)      ->  "Hello world"      (exactly fits)
shorten("Hello wonderful world", 15)  ->  "Hello..."   (cut at a space: never mid-word)
shorten("Supercalifragilistic", 10)   ->  "Superca..." (one long word: cut mid-word after all)
shorten("Hi", 2)                ->  "Hi"
shorten("Hello", 3)             ->  "..."
```

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_examples():\n    \"the examples\"\n    assert shorten(\"Hello world\", 20) == \"Hello world\"\n    assert shorten(\"Hello world\", 8) == \"Hello...\"\n    assert shorten(\"Hello world\", 11) == \"Hello world\"\n    assert shorten(\"Hello wonderful world\", 15) == \"Hello...\"\n    assert shorten(\"Supercalifragilistic\", 10) == \"Superca...\"\n    assert shorten(\"Hi\", 2) == \"Hi\"\n    assert shorten(\"Hello\", 3) == \"...\"\ndef test_more():\n    \"more cases\"\n    assert shorten(\"a b c d e f\", 7) == \"a b...\"\n    assert shorten(\"one two\", 6) == \"one...\"")

### Exercise: Checks before code

Before anyone writes `median(values)`, write the checks. Write `median_checks()`, which uses `assert` to check `median` on worked examples: an odd-length list, an even-length list (the mean of the middle two), an unsorted list, a list with repeats, a one-item list, negative numbers, and that it doesn't change the list it's given. The checker runs your checks against a correct `median` and against four broken ones.

In [ ]:
def median(values):
    s = sorted(values)
    mid = len(s) // 2
    return s[mid] if len(s) % 2 else (s[mid - 1] + s[mid]) / 2

def median_checks():
    ___

median_checks()
print("checks pass")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def b_unsorted(values):\n    mid = len(values) // 2\n    return values[mid] if len(values) % 2 else (values[mid - 1] + values[mid]) / 2\ndef b_even(values):\n    s = sorted(values)\n    return s[len(s) // 2]\ndef b_mutates(values):\n    values.sort()\n    mid = len(values) // 2\n    return values[mid] if len(values) % 2 else (values[mid - 1] + values[mid]) / 2\ndef b_int(values):\n    s = sorted(values)\n    mid = len(s) // 2\n    return s[mid] if len(s) % 2 else (s[mid - 1] + s[mid]) // 2\ndef test_pass():\n    \"pass with the real median\"\n    median_checks()\ndef test_catch():\n    \"catch the broken ones\"\n    g = median_checks.__globals__\n    real = g[\"median\"]\n    missed = []\n    for name, fn in [(\"ignores order\", b_unsorted), (\"even length wrong\", b_even), (\"changes the list\", b_mutates), (\"rounds the even case down\", b_int)]:\n        g[\"median\"] = fn\n        try:\n            median_checks()\n            missed.append(name)\n        except Exception:\n            pass\n        finally:\n            g[\"median\"] = real\n    assert not missed, \"Your checks didn't catch: \" + \", \".join(missed)")

### Bug hunt: The wrong question

The brief: *"Return how many days are left until the end of the year, not counting today."* The function below was written for a different question. Using the examples, work out what it gets wrong, and fix it. `date` objects come from the `datetime` module, and subtracting two gives a difference whose `.days` is a whole number.

In [ ]:
from datetime import date

def days_left(today):
    end = date(today.year, 12, 31)
    return (end - today).days + 1

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import date\ndef test_values():\n    \"examples\"\n    assert days_left(date(2026, 12, 31)) == 0\n    assert days_left(date(2026, 12, 30)) == 1\n    assert days_left(date(2026, 1, 1)) == 364\n    assert days_left(date(2024, 1, 1)) == 365")

### Exercise: Work it by hand

Use the rules from today, working by hand, not by running code.

In [ ]:
a = ___   # luhn_ok("18") (True or False)
b = ___   # clock_diff("22:15", "06:45") in minutes
c = ___   # next_term("312211")
d = ___   # the 6174 step from 6174 itself: 7641 - 1467

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a is True\n    assert b == 510\n    assert c == \"13112221\"\n    assert d == 6174")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the next date

Write `next_day(text)` that takes a date written `YYYY-MM-DD` and returns the next day's date in the same format, **without** using the `datetime` module. Raise a `ValueError` (message `invalid date: ` and the text in quotes) for anything that isn't a real date from year 1 to 9999. Work examples by hand first: an ordinary day, the end of a 30-day month, the end of February in a leap year and in a non-leap year (remember the century rule), and New Year's Eve.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def bad(t):\n    try:\n        next_day(t)\n    except ValueError as e:\n        assert str(e) == f\"invalid date: {t!r}\"\n        return\n    raise AssertionError(f\"{t!r} should raise a ValueError\")\ndef test_values():\n    \"several\"\n    assert next_day(\"2026-10-27\") == \"2026-10-28\"\n    assert next_day(\"2026-04-30\") == \"2026-05-01\"\n    assert next_day(\"2024-02-28\") == \"2024-02-29\"\n    assert next_day(\"2024-02-29\") == \"2024-03-01\"\n    assert next_day(\"2026-02-28\") == \"2026-03-01\"\n    assert next_day(\"1900-02-28\") == \"1900-03-01\"\n    assert next_day(\"2000-02-28\") == \"2000-02-29\"\n    assert next_day(\"2026-12-31\") == \"2027-01-01\"\n    assert next_day(\"0009-12-31\") == \"0010-01-01\"\ndef test_bad():\n    \"invalid\"\n    for t in [\"2026-02-29\", \"2026-13-01\", \"2026-04-31\", \"2026-1-01\", \"2026/10/27\", \"\", \"9999-12-31\", \"0000-01-01\"]:\n        bad(t)\ndef test_all():\n    \"every day of four years\"\n    from datetime import date, timedelta\n    d = date(1999, 1, 1)\n    while d < date(2003, 1, 1):\n        assert next_day(d.isoformat()) == (d + timedelta(days=1)).isoformat()\n        d += timedelta(days=1)\n    assert \"datetime\" not in source")

## A method

In 1945 the mathematician George Pólya wrote a small book called *How to Solve It*, about how to approach a problem you don't yet know how to solve. Programmers still use its four steps, which this week is built around:

1. **Understand the problem.** What is given, what is wanted, and what are the rules? (Today.)
2. **Make a plan.** Find a way from what's given to what's wanted: a similar problem you've solved, a pattern, a way to split it into smaller problems. (Days 3 to 5.)
3. **Carry out the plan**, checking each step.
4. **Look back.** Check the result, test the awkward cases (Day 2), and ask what you've learned for next time.

The step that beginners skip is the first, because it doesn't feel like progress. It is: a few minutes understanding a problem saves hours of debugging a solution to the wrong one.

## Inputs, outputs and rules

Here's a problem, as it might be given to you:

> Write a function that checks whether a credit card number is valid.

That isn't something you can program yet. **What** goes in? A number? A string with spaces? **What** comes out? `True` or `False`, or a message? **What** makes it valid? You need the rule.

Card numbers use the **Luhn check**, invented by Hans Peter Luhn at IBM in the 1950s, which catches most single-digit typing mistakes. Here's the rule:

> Starting from the rightmost digit and moving left, double every second digit (so the second from the right, the fourth from the right, and so on). If doubling gives more than 9, subtract 9. Add up all the digits, doubled and not. The number is valid if the total is a multiple of 10.

Now you can write the problem down properly:

| | |
|---|---|
| **Input** | a string of digits, possibly containing spaces (as people type card numbers) |
| **Output** | `True` if it passes the Luhn check, otherwise `False` |
| **Rules** | ignore spaces; anything else that isn't a digit makes it invalid; fewer than 2 digits is invalid; then the Luhn rule |

Notice the rows that weren't in the original sentence: spaces, other characters, very short input. Writing the table **made you decide**. You might decide differently (raise an error for letters, say), and that's fine, as long as it's a decision, not an accident.

## Examples by hand

Before coding, work an example on paper. Take `79927398713`, the example Luhn himself used:

| Digit (from the right) | 3 | 1 | 7 | 8 | 9 | 3 | 7 | 2 | 9 | 9 | 7 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| Double? | no | yes | no | yes | no | yes | no | yes | no | yes | no |
| Value | 3 | 2 | 7 | 16 → 7 | 9 | 6 | 7 | 4 | 9 | 18 → 9 | 7 |

Total: 3 + 2 + 7 + 7 + 9 + 6 + 7 + 4 + 9 + 9 + 7 = 70, a multiple of 10, so it's valid.

Working it by hand does three things. It checks you understand the rule. It tells you the **steps** your program needs (reverse, double every second one, subtract 9, add up). And it gives you an **example with a known answer** to check your program against.

Every line of that function comes straight from the worked table. That's what working an example first buys you.

## Finding the gaps

Real problem descriptions are incomplete. Working examples is how you find the gaps. Here's another:

> Write a function that returns the time difference between two clock times.

Work some examples, and the questions appear:

- `"09:00"` and `"17:30"`: 8 hours 30 minutes. In what form: `510` minutes, `"8:30"`, `(8, 30)`?
- `"17:30"` and `"09:00"`: is that minus 8:30, or 15:30 (from 5.30 pm to 9 the next morning)?
- `"23:00"` and `"01:00"`: probably 2 hours, across midnight. Is that the same rule as the previous one?
- `"9:5"`: allowed? `"24:00"`?

None of these are hard to program. All of them are decisions, and each one would be a bug if you guessed differently from what was meant. In real work you'd **ask**. In an exercise, the examples given usually answer them; read them closely.

### Questions to ask every time

- What exactly are the inputs: types, ranges, formats? Can they be empty?
- What exactly is the output: type, format, order?
- What happens at the edges: zero, one, none, the largest, ties, duplicates?
- What should happen with bad input: an error, a special value, or can I assume it won't happen?
- Is there an example in the description? Do my rules give the same answer for it?

## Examples become checks

Every worked example is a ready-made check. Write them **before** the function, and you'll know the moment it works, and the moment a later change breaks it:

The `% (24 * 60)` handles midnight in one step. From 23:00 to 01:00, the plain difference is 60 - 1380 = -1320 minutes, and `-1320 % 1440` is `120`. Python's `%` always gives a result from 0 up to (but not including) the divisor, which is exactly "going forward round the clock". The examples confirmed it before we had to trust the reasoning.

## Practice

In each exercise, work an example by hand **before** coding. The descriptions are complete, but read every sentence: each one matters.

## Quiz

## Daily challenge

## Before you go

Take any exercise from Phase 1 that you found hard, and write its inputs, outputs and rules in a table, with two worked examples, as if you'd been given only one sentence. Notice how many decisions the exercise had already made for you. Tomorrow: edge cases, the inputs where most bugs live.